# Building an NL to SQL Pipeline

Turn an English question into a SQL query, run it on a database, and get an English answer back. We will:

1. Look at the data we have.
2. Write down a fixed test suite (questions, expected SQL, expected answers).
3. Build the pipeline step by step.
4. Build the **system prompt itself** in three versions and watch the test suite tell us which version actually works.

The pipeline has six stages:

```
question  ->  generate SQL  ->  validate  ->  execute  ->  summarize  ->  answer
                  (LLM)         (Python)    (SQLite)        (LLM)
```

Two LLM calls. One DB call.


## Setup

> **Don't have `uv`?** Install it once with:
> ```
> curl -LsSf https://astral.sh/uv/install.sh | sh
> ```
> Windows users: see <https://docs.astral.sh/uv/getting-started/installation/>.


In [2]:
# uv add reads pyproject.toml in this folder and installs into the project venv.
# If you cloned this repo, run this once. If you're in Colab, replace with:
#     !pip install langchain-openai langchain-core pandas --quiet
!pip install langchain-openai langchain-core pandas

^C


In [6]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

In [15]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("OPENAI_API_KEY"))

In [22]:
import sqlite3
from pathlib import Path

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

DB = Path("spacex_launches.db")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## 1. Look at the data first

One table called `launches` with 18 rows of real-ish SpaceX missions. Scan the rows and you can answer most questions by eye.


In [ ]:
con = sqlite3.connect(DB)
df = pd.read_sql_query("SELECT * FROM launches ORDER BY launch_date", con)
con.close()
df

## 2. The test suite

Before we write any pipeline code, we write down **what success looks like**. For each question, we record three things:

1. The user **question**.
2. The **SQL** we expect a correct pipeline to produce (or close to it).
3. The **answer text** we expect the user to see (a substring is enough).

Maintaining this table is the most useful habit you can pick up in NL-to-SQL work. Without it, you cannot tell if a prompt change made things better or worse.


In [ ]:
test_suite = [
    {
        "question": "How many launches were successful?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE success = 1",
        "expected_answer": "14",
    },
    {
        "question": "What was the heaviest payload ever launched?",
        "expected_sql": "SELECT mission_name, payload_kg FROM launches ORDER BY payload_kg DESC LIMIT 1",
        "expected_answer": "Starlink",
    },
    {
        "question": "Which vehicle has the highest success rate?",
        "expected_sql": "SELECT vehicle, ... GROUP BY vehicle ORDER BY ... DESC LIMIT 1",
        "expected_answer": "Falcon Heavy",
    },
    {
        # The DIFFERENTIATING test case.
        # 'heavy launch' looks obvious - the LLM will likely match it to the literal
        # `vehicle = 'Falcon Heavy'` value it sees in the schema (giving 2).
        # But our business defines a 'heavy launch' as `payload_kg > 5000` regardless
        # of vehicle. The correct answer is 6. Without this term in schema.md the LLM
        # gets it confidently wrong.
        "question": "How many heavy launches have we flown?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE payload_kg > 5000",
        "expected_answer": "6",
    },
    {
        # Another domain term the LLM cannot guess: 'civilian mission' = customer = 'Shift4'
        "question": "How many civilian missions have flown?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE customer = 'Shift4'",
        "expected_answer": "2",
    },
]

pd.DataFrame(test_suite)

## 3. What SQLite already knows about itself

SQLite stores every table's DDL in a meta-table called `sqlite_master`. We can ask it for the exact `CREATE TABLE` statement. This is the bare minimum a database can hand us.


In [ ]:
con = sqlite3.connect(DB)
ddl = con.execute(
    "SELECT sql FROM sqlite_master WHERE type='table' AND name='launches'"
).fetchone()[0]
con.close()

print(ddl)

That is **everything** the database itself can tell us. Notice what is missing:

- Allowed values for text columns. Is the rocket called `'Falcon 9'` or `'Falcon-9'`?
- **Domain terms.** What does **"heavy launch"** mean in our business? What does **"civilian mission"** mean?
- Boolean meaning. We see `CHECK (success IN (0, 1))` but not which one means succeeded.
- Conventions. We see `payload_kg` is `INTEGER NOT NULL` but not that **`0` means no orbital payload**.

The LLM will have to **guess** all of that. Sometimes it guesses right. Sometimes it confidently guesses wrong, which is the failure mode we will demo in section 6.


## 4. Three versions of the system prompt

We will build three prompts and run the **same test suite** through each one.

- **v1: DDL only** - just what `sqlite_master` gave us above.
- **v2: + column descriptions, allowed values, conventions** - everything we listed as "missing" except domain terms and few-shot examples.
- **v3: + domain terms + few-shot examples** - the full `schema.md` file.


In [ ]:
# Prompt v1: DDL only ----------------------------------------------------------
prompt_v1 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL only):
{ddl}
"""

print("--- prompt_v1 ---")
print(prompt_v1)

In [ ]:
# Prompt v2: + column descriptions and conventions -----------------------------
column_descriptions = """
Column descriptions:
- launch_id     (INTEGER): primary key, not meaningful.
- mission_name  (TEXT):    human-readable name like 'Crew Dragon Demo-2'.
- vehicle       (TEXT):    rocket family. Allowed values:
                           'Falcon 1', 'Falcon 9', 'Falcon Heavy', 'Starship'.
- payload_type  (TEXT):    Allowed values:
                           'Test', 'Cargo', 'Crew', 'Satellite', 'Probe'.
- payload_kg    (INTEGER): mass of payload in kg.
                           **0 means the launch had NO orbital payload (test flight).**
- destination   (TEXT):    examples:
                           'LEO', 'ISS', 'L1', 'Heliocentric', 'Suborbital', 'Jupiter Transfer'.
- success       (INTEGER): **1 = succeeded, 0 = failed.** Treat as boolean.
- launch_date   (DATE):    YYYY-MM-DD.
- customer      (TEXT):    examples:
                           'NASA', 'SpaceX', 'NOAA', 'Shift4', 'Iridium', 'MDA', 'DARPA'.

Conventions:
- Internal SpaceX flights have customer = 'SpaceX'.
- Test flights with no orbital payload have payload_kg = 0.
"""

prompt_v2 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL):
{ddl}

{column_descriptions}
"""

print("--- prompt_v2 (first 600 chars) ---")
print(prompt_v2[:600])
print("...")

In [ ]:
# Prompt v3: + domain terms + few-shot (full schema.md) -----------------------
SCHEMA_DOC = Path("schema.md").read_text()

prompt_v3 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

{SCHEMA_DOC}
"""

print(f"prompt_v1: {len(prompt_v1)} chars")
print(f"prompt_v2: {len(prompt_v2)} chars")
print(f"prompt_v3: {len(prompt_v3)} chars  (loaded from schema.md, includes domain terms and few-shot)")

## 5. Build the pipeline once

We bundle the four pipeline stages into one tiny helper. The body is exactly the linear code you would write yourself - no inheritance, no decorators, no magic. Just: ask LLM for SQL, validate, run it, ask LLM for an answer.


In [ ]:
SUMMARY_PROMPT = (
    "Answer the user's question in one short sentence using only the rows. "
    "Use digits (e.g. '14', not 'fourteen') and cite numbers exactly. "
    "Do not speculate. "
    "If rows is empty, say you couldn't find an answer."
)

FORBIDDEN = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE")


def run_pipeline(system_prompt, question):
    # (1) generate SQL
    sql = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ]).content.strip()
    if sql.startswith("```"):
        sql = sql.split("```")[1]
        if sql.lstrip().lower().startswith("sql"):
            sql = sql.lstrip()[3:]
        sql = sql.strip()

    # (2) validate
    upper = sql.upper()
    for w in FORBIDDEN:
        if w in upper:
            return {"sql": sql, "rows": None, "answer": f"REFUSED: contains {w}"}
    if "FROM LAUNCHES" not in upper.replace("\n", " "):
        return {"sql": sql, "rows": None, "answer": "REFUSED: must read from launches"}
    aggs = ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")
    if "LIMIT" not in upper and not any(a in upper for a in aggs):
        sql = sql.rstrip(";\n ") + " LIMIT 100"

    # (3) execute
    try:
        con = sqlite3.connect(DB)
        con.row_factory = sqlite3.Row
        rows = [dict(r) for r in con.execute(sql).fetchall()]
        con.close()
    except Exception as e:
        return {"sql": sql, "rows": None, "answer": f"SQL ERROR: {e}"}

    # (4) summarize
    answer = llm.invoke([
        SystemMessage(content=SUMMARY_PROMPT),
        HumanMessage(content=f"Question: {question}\nRows: {rows}"),
    ]).content.strip()
    return {"sql": sql, "rows": rows, "answer": answer}

## 6. Detailed walkthrough on one tricky question

Let's pick the question that should clearly separate the three prompt versions:

> **"How many heavy launches have we flown?"**

Read that question through the eyes of the LLM. The schema has a column `vehicle` and one of its allowed values is `'Falcon Heavy'`. The word **heavy** appears literally in the data. The LLM will see this and assume "heavy launch" means a launch of the Falcon Heavy rocket. So it will write:

```sql
SELECT COUNT(*) FROM launches WHERE vehicle = 'Falcon Heavy';   -- returns 2
```

But that is **not what our business means by "heavy launch"**. Our terminology is:

> **"heavy launch" = any mission with payload over 5,000 kg, regardless of which rocket flew it.**

The correct SQL is:

```sql
SELECT COUNT(*) FROM launches WHERE payload_kg > 5000;          -- returns 6
```

The LLM cannot guess this. The word "heavy" does not mean "Falcon Heavy" in our domain. It has to be **written down** in our schema documentation. Watch the three prompts argue about it.


In [ ]:
tricky_q = "How many heavy launches have we flown?"
expected_sql = "SELECT COUNT(*) FROM launches WHERE payload_kg > 5000"
expected_answer = "6"

print(f"Question:        {tricky_q}")
print(f"Expected SQL:    {expected_sql}")
print(f"Expected answer: contains '{expected_answer}' (i.e. 6 missions)")

### Run with prompt v1 (DDL only)

The LLM sees `'Falcon Heavy'` as one of the allowed vehicle values. It will probably match "heavy" to that value and return 2. **Confidently wrong.**


In [ ]:
r1 = run_pipeline(prompt_v1, tricky_q)
print(f"v1 SQL:    {r1['sql']}")
print(f"v1 rows:   {r1['rows']}")
print(f"v1 answer: {r1['answer']}")
print(f"v1 PASS?   {expected_answer in r1['answer']}")

### Run with prompt v2 (DDL + descriptions + conventions)

v2 spells out the allowed values for `vehicle`. That actually makes the wrong interpretation **stronger** - the LLM sees `'Falcon Heavy'` even more clearly and will match "heavy" to it. Same answer, same wrong answer.


In [ ]:
r2 = run_pipeline(prompt_v2, tricky_q)
print(f"v2 SQL:    {r2['sql']}")
print(f"v2 rows:   {r2['rows']}")
print(f"v2 answer: {r2['answer']}")
print(f"v2 PASS?   {expected_answer in r2['answer']}")

### Run with prompt v3 (full schema.md, including the 'heavy launch' domain term)

v3 has the term defined explicitly in the **Domain terms** table inside `schema.md`:

> `heavy launch` -> `payload_kg > 5000`

Now the LLM uses our definition, not its assumption.


In [ ]:
r3 = run_pipeline(prompt_v3, tricky_q)
print(f"v3 SQL:    {r3['sql']}")
print(f"v3 rows:   {r3['rows']}")
print(f"v3 answer: {r3['answer']}")
print(f"v3 PASS?   {expected_answer in r3['answer']}")

**This is why the schema doc matters.**

- The DDL tells you the columns and types. Fine for simple questions.
- Column descriptions and conventions (v2) tell the LLM what each column *is*. They lock down spelling and allowed values, but they do not define **what your business calls things**.
- Only when we write down domain terms (`heavy launch`, `civilian mission`, `tier-1 customer`, `churned user`, etc.) does the LLM stop guessing.

The lesson: **the LLM will confidently produce the wrong answer when business vocabulary collides with column values**. The fix is not better LLMs. The fix is writing down what your terms mean.


## 7. Run the full test suite across all three prompts

Now run every test case through every prompt version. This is the table that tells you whether a prompt change helped or hurt.


In [ ]:
def grade(prompt_label, prompt, suite):
    rows = []
    for case in suite:
        r = run_pipeline(prompt, case["question"])
        passed = case["expected_answer"].lower() in r["answer"].lower()
        rows.append({
            "prompt": prompt_label,
            "question": case["question"][:45] + ("..." if len(case["question"]) > 45 else ""),
            "expected": case["expected_answer"],
            "sql_generated": " ".join(r["sql"].split())[:80] + ("..." if len(r["sql"]) > 80 else ""),
            "answer": r["answer"][:70] + ("..." if len(r["answer"]) > 70 else ""),
            "pass": "PASS" if passed else "FAIL",
        })
    return rows


grades_v1 = grade("v1: DDL only",       prompt_v1, test_suite)
grades_v2 = grade("v2: + descriptions", prompt_v2, test_suite)
grades_v3 = grade("v3: + few-shot",     prompt_v3, test_suite)

all_grades = pd.DataFrame(grades_v1 + grades_v2 + grades_v3)
all_grades

In [ ]:
# A compact pass-rate summary
summary = pd.DataFrame({
    "question": [c["question"][:40] + ("..." if len(c["question"]) > 40 else "") for c in test_suite],
    "expected": [c["expected_answer"] for c in test_suite],
    "v1: DDL only":       [g["pass"] for g in grades_v1],
    "v2: + descriptions": [g["pass"] for g in grades_v2],
    "v3: + few-shot":     [g["pass"] for g in grades_v3],
})

print(f"v1 (DDL only):           {sum(1 for g in grades_v1 if g['pass']=='PASS')} / {len(test_suite)}")
print(f"v2 (+ descriptions):     {sum(1 for g in grades_v2 if g['pass']=='PASS')} / {len(test_suite)}")
print(f"v3 (+ few-shot):         {sum(1 for g in grades_v3 if g['pass']=='PASS')} / {len(test_suite)}")
print()
summary

**The pattern you should see:** trivial questions pass on all three. Questions that depend on **domain terms** (heavy launch, civilian mission) fail on v1 and v2 and only pass on v3.

This is the production reality: a strong LLM gets the easy 80% right with no help. The remaining 20% requires you to **write down what your business terms mean**. The test suite is what tells you when you have written enough.


---

## 8. Full pipeline walkthrough on a single question (using v3)

Now that we know v3 is the prompt we want, let's trace one question end-to-end with all the intermediate state visible.


In [ ]:
# (1) the user's question
question = "Which vehicle has the highest success rate?"
print(f"USER QUESTION:\n  {question}\n")

In [ ]:
# (2) ask the LLM to write the SQL
sql_response = llm.invoke([
    SystemMessage(content=prompt_v3),
    HumanMessage(content=question),
])
sql = sql_response.content.strip()
if sql.startswith("```"):
    sql = sql.split("```")[1]
    if sql.lstrip().lower().startswith("sql"):
        sql = sql.lstrip()[3:]
    sql = sql.strip()

print(f"GENERATED SQL:\n{sql}\n")

In [ ]:
# (3) validate
upper = sql.upper()
for w in FORBIDDEN:
    assert w not in upper, f"refused: {w}"
assert "FROM LAUNCHES" in upper.replace("\n", " "), "refused: must read from launches"
aggs = ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")
if "LIMIT" not in upper and not any(a in upper for a in aggs):
    sql = sql.rstrip(";\n ") + " LIMIT 100"

print(f"VALIDATED SQL:\n{sql}\n")

In [ ]:
# (4) execute on SQLite
con = sqlite3.connect(DB)
con.row_factory = sqlite3.Row
rows = [dict(r) for r in con.execute(sql).fetchall()]
con.close()

print("ROWS FROM DB:")
for r in rows:
    print(" ", r)

In [ ]:
# (5) ask the LLM to write the English answer
answer_response = llm.invoke([
    SystemMessage(content=SUMMARY_PROMPT),
    HumanMessage(content=f"Question: {question}\nRows: {rows}"),
])
answer = answer_response.content.strip()

print(f"FINAL ANSWER:\n  {answer}")

---

## 9. Three failure modes worth seeing


### Failure 1: validator catches dangerous SQL


In [ ]:
r = run_pipeline(prompt_v3, "Delete all the failed launches")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

### Failure 2: question is out of scope (the database has no answer)


In [ ]:
r = run_pipeline(prompt_v3, "What year did SpaceX go public?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

### Failure 3: ambiguity is interpreted, not refused

There is no rocket called "Falcon X". The LLM does not refuse. It silently reinterprets the question and gives you a confidently wrong answer. Look at the SQL.


In [ ]:
r = run_pipeline(prompt_v3, "How many Falcon X missions have flown?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

---

## 10. What this pipeline cannot do

This is a **pipeline**, not an agent. Six fixed steps in fixed order.

| Question                                                              | Why this pipeline fails                                  |
|-----------------------------------------------------------------------|----------------------------------------------------------|
| "What did Elon say about CRS-7 in the press release?"                 | We have no documents to search.                          |
| "Compare these missions to the Apollo program."                       | Apollo is not in our database. SQL returns empty rows.   |
| "First it was 14 successes. Now compare that to the total."           | No memory between calls. Each call starts fresh.         |
| "I think the SQL was wrong. Try a different query."                   | No retry. No reflection. One shot.                       |
| "Find missions similar to Inspiration4."                              | Wrong tool. We would need vector search, not SQL.        |
| "Show me a chart of launches per year."                               | Pipeline returns text. No plotting tool.                 |

The next class introduces **agents**, which can decide which tool to use, retry on failure, and remember earlier turns. That is how we address the limitations above.


---

## 11. Try it yourself

1. Add a new test case to `test_suite` that depends on a domain term we have not defined. Run the suite. Does any prompt pass it?
2. Add the missing domain term to `schema.md`, restart the kernel, and re-run. Does the test pass now?
3. Replace `gpt-4o-mini` with `gpt-3.5-turbo` and re-run the suite. Which questions now fail on v1?
4. Pick one FAIL row. Look at the SQL the LLM produced. What specifically did the prompt fail to communicate?
5. Add `print(sql)` inside `run_pipeline` so every call shows its SQL, then ask three new questions of your own.
